# **Project work: A mini segmentation challenge**

<div style="color:#777777;margin-top: -15px;">
<b>Course</b>: MSLS / CO4 |
<b>Version</b>: v1.3 <br><br>
<!-- 10.04.2025, v1.2: Fully refactored -->
<!-- 17.04.2026, v1.3: Introduced word count limits -->
</div>


**Student**: $\Rightarrow$ Livia Vizens, Cyrille Niklaus

**Email**: $\Rightarrow$ vinzeliv@students.zhaw.ch, niklacyr@students.zhaw.ch

**University**: $\Rightarrow$  ZHAW

**Semester**: $\Rightarrow$  2nd Semester

**Date**: $\Rightarrow$  DATE OF SUBMISSION


<br>

## **Abstract**

*This project develops a classical image segmentation pipeline for E. coli colony detection on self-made petri dishes. We then evaluate the generalizability of our method by testing it on images from the AGAR dataset, assessing how well our segmentation strategy transfers across different imaging conditions and experimental setups. This comparison helps identify whether our approach can reliably handle diverse colony images beyond our controlled lab environment. Further, this project acts as a benchmark segmentation and bounding box approach for a deep learning applications related to the Project work of the module "Deep Learning".*



<br><br>

----

## **Table of contents**
<!-- Unfortunately, the following does not always work correctly -->
* [1. Dataset](#sec_dataset)  
* [2. Preprocessing](#sec_preprocessing)  
* [3. Manual segmentation](#sec_manual_segmentation)  
* [4. Automated segmentation](#sec_automated_segmentation)  
* [5. Evaluation](#sec_evaluation)  
* [6. Discussion](#sec_discussion)  
* [Appendix: Hints](#sec_hints)  


---

## **Prerequisites / Setup**

$\Rightarrow$ Special setup instructions, imports and configurations go here.


In [64]:
import numpy as np
import matplotlib.pyplot as plt
import cv2 as cv
import PIL
from PIL import Image
from rembg import remove
from pandas.core.dtypes import astype


# Jupyter / IPython configuration:
# Automatically reload modules when modified
%load_ext autoreload
%autoreload 2

# Enable vectorized output (for nicer plots)
%config InlineBackend.figure_formats = ["svg"]

# Inline backend configuration
%matplotlib inline

# Enable this line if you want to use the interactive widgets
# It requires the ipympl package to be installed.
#%matplotlib widget

import sys
sys.path.insert(0, "../")
import tools


No onnxruntime backend found.
Please install rembg with CPU or GPU support:

    pip install "rembg[cpu]"  # for CPU
    pip install "rembg[gpu]"  # for NVIDIA/CUDA GPU

For more information, see: https://github.com/danielgatis/rembg#installation


SystemExit: 1

---


<a id='sec_dataset'></a>

## **Dataset**

*$\Rightarrow$ Describe your dataset.*

### **Requirements**
* Provide a dataset with at least 10 image samples.
* The dataset must be no larger than 200 MB. If it exceeds this size, please contact the tutor in advance.
* Ensure that you have the rights to use and share the data (check the usage license).
* The images should clearly show a recognizable structure of interest.
* Avoid datasets with too much variation.
* Each student/team must use a different dataset!



---

<a id='sec_preprocessing'></a>

## **Preprocessing**

*$\Rightarrow$ Describe the pre-processing steps applied to enhance the input images.*

*Note: The specific steps will depend on your dataset and the intended application.*

### **Instructions:**
* Improve image quality by reducing noise, adjusting contrast, or normalizing intensity.
* Standardize image dimensions and formats for consistent input to analysis pipelines.
* Highlight or isolate relevant structures to support downstream processing.




### **Image Cropping:**
* Our self-made petri dished are imaged by using a scanner.
* In order to separate the single petri dishes into one image each, we crop the original scan.
* One petri dish has the diensions 4150, 4150, 3
* The images are combined to a list, for further processing **(to implement)**



In [ ]:
# Read test image
test_img = cv.imread("high_res_color_restore.tif", cv.IMREAD_COLOR)
# Change BGR to RGB representation
test_img = cv.cvtColor(test_img, cv.COLOR_BGR2RGB)
# crop the petri dish
test_img_cropped = test_img[4700:8850, 3450:7600].copy()
# print some stats
print(test_img_cropped.shape) # 4150, 4150 -> dimensions of a petri dish
print(test_img_cropped.dtype)
# Show our test images
plt.imshow(test_img) # used this because you can see the pixel range of the whoel image
plt.show()
#tools.show_image(test_img)
tools.show_image(test_img_cropped)

### **Gray-scaling & Resizing:**
* The imgeas in the image list are processed to a weighted gray-scaled images. Then, they are resized, in order to lower computation.
* They are returned as np.array
* Below you can find some illustrations about how the chromatic variance between petri dishes influences the grayscale.
* For green/blueish background, gray-scaling improved the differentiation between background and colony.




In [ ]:
img_to_process = [test_img_cropped]
def process_image(img, size=(1024, 1024), to_grayscale=True):
    """Process image: convert to grayscale, resize, return as NumPy array."""
    if to_grayscale:
        img = (img @ [0.299, 0.587, 0.114]).astype(np.uint8)
    return np.array(PIL.Image.fromarray(img).resize(size))

# Process list (if several images to process)
imgs_list = [process_image(i) for i in img_to_process]

# Process test images (to illustrate the difference) -> gray much better for differentiation
gray_cropped_resized = process_image(test_img_cropped)
color_cropped_resized = process_image(test_img_cropped, to_grayscale=False)

# Display images and ROIs
tools.show_image_pair(gray_cropped_resized, color_cropped_resized)
tools.show_image_pair(gray_cropped_resized[200:250, 200:250], color_cropped_resized[200:250, 200:250])

### **Histogram:**
* Below we see the distribution of shades of gray in our image.
* We see a the majority is between 130 and 170.
* We illustraded how equalization decreases the ability to differentiate between colony and background.




In [ ]:
hist, bins = np.histogram(gray_cropped_resized.flatten(), bins=256, range=[0,256], density=True)
cdf = hist.cumsum()
hist /= hist.max()

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

axes[0].imshow(gray_cropped_resized, cmap="gray")
axes[0].axis("off")
axes[1].plot(hist, label="Histogram (normalized)")
axes[1].plot(cdf, label="CDF")
axes[1].set_xlabel("Pixel value")
axes[1].legend();


# Apply histogram equalization
gray_equalized = cv.equalizeHist(gray_cropped_resized.copy())
hist_equalized, bins = np.histogram(gray_equalized.flatten(),
                                    bins=256, range=[0,256],
                                    density=True)
cdf_equalized = hist_equalized.cumsum()
hist_equalized /= hist_equalized.max()
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(gray_equalized, cmap="gray")
axes[0].axis("off")
axes[1].plot(hist_equalized, label="Histogram (normalized)")
axes[1].plot(cdf_equalized, label="CDF")
axes[1].set_xlabel("Pixel value")
axes[1].legend();

# No improvement when applying equalization.
tools.show_image_pair(gray_cropped_resized[200:250, 200:250], gray_equalized[200:250, 200:250])

### **Masking:**
* Since the petri dish in the image is round, the corner parts, displaying only background, are not necessary.
* We masked the background for further image processing.

In [ ]:
def background_masking(img, cx, cy, r):
    mask = np.zeros(img.shape[:2], dtype=np.uint8)
    mask = cv.circle(mask, (cy, cx), r, 255, -1)
    mask = mask.astype(bool)
    img_masked = img.copy()
    img_masked[~mask] = 0
    return img_masked

"""
# Process list of images
imgs_masked = [background_masking(i, 512, 512, 512) for i in imgs_list]
"""

img2_masked = background_masking(gray_cropped_resized, 512, 512, 512)

tools.show_image_pair(gray_cropped_resized, img2_masked, title1="Original", title2="Masked")

### *Denoising*
* trying out denoising (bilateral to preserve the edges
* although visually a pain to look at the whole petri dish. In the zoomed in version we see that the shade distribution of the colonies improved. They appear now much more uniform.

In [ ]:
def blur(img, d, sigmaColor, sigmaSpace):
    img_blur = cv.bilateralFilter(img, d, sigmaColor, sigmaSpace)
    return img_blur

"""
imgs_blurred = [blur(i, 7, 75, 75) for i in imgs_masked]
"""

img_mask_blur = blur(gray_cropped_resized, 7, 75, 75)

tools.show_image_pair(gray_cropped_resized[200:250, 200:250], img_mask_blur[200:250, 200:250])

### *Sharpening*

* in order to enhance image edges we apply a Laplacian Filter
* unsure about outcome. We see an enhanced edge. But will it help?

In [ ]:
kernel = [[0, 1, 0], [1, -4, 1], [0, 1, 0]]
kernel = np.asarray(kernel)
#kernel1 = [[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]]
#kernel1 = np.asarray(kernel1)

def sharp(img, kernel):
    img_sharp = cv.filter2D(img, ddepth=-1, kernel=kernel)
    img_sharp = np.abs((img-img_sharp))
    return img_sharp

img_sharp = sharp(img_mask_blur, kernel)

"""
imgs_sharp = [sharp(i, kernel] for i in imgs_blurred]
"""

tools.show_image_pair(img_mask_blur[200:250, 200:250], img_sharp[200:250, 200:250],
                      title1= "img_mask_blur",
                      title2="img_sharp")

### Background removal

* Try to remove background using the RemBG package

---

<a id='sec_manual_segmentation'></a>

## **Manual segmentation**

*$\Rightarrow$ Describe the manual segmentation step*


### **Instructions:**
* Use a suitable tool to manually segment the structures of interest.  
* These segmentations will be needed for further analysis (or model training).  
* If your dataset already includes segmentation masks, you still need to demonstrate how such masks can be created manually.

---

<a id='sec_automated_segmentation'></a>

## **Automated Segmentation**

*$\Rightarrow$ Describe how the images are segmented using Python.*

### **Instructions:**
* Perform the segmentation in Python.
* You may use external libraries or tools (e.g., OpenCV, scikit-image).
* Implement a function `segment(image, ...)` that takes an image as input and returns a segmentation mask for the structure of interest.


---

<a id='sec_evaluation'></a>

## **Evaluation**

*$\Rightarrow$ Describe the evaluation of your results.*

### **Instructions:**
* Select an evaluation method to compare two binary segmentation masks and quantify how well they match (e.g., using the Dice score).
* Hint: Implement a function `evaluate(mask1, mask2)` that returns the chosen evaluation score(s).
* Calculate the mean and standard deviation of the scores across the entire dataset.



---

<a id='sec_discussion'></a>

## **Discussion**

*$\Rightarrow$ Briefly discuss your results and share your key observations and your experiences and leaernings.*



---

<a id='sec_references'></a>

## **References**

*$\Rightarrow$ List all relevant references (as URLs).*

*Also, clearly state whether you used generative AI tools (e.g., ChatGPT, GitHub Copilot) and describe how they were used.*



<br><br><br><br><br><br><br><br>

---

<a id='sec_hints'></a>

## **Appendix: Hints**

### **Markdown / HTML**

The following tutorials might be useful if you are not yet familiar with Markdown:

- [Quick overview](https://www.writethedocs.org/guide/writing/markdown/)
- [Markdown GitHub-style](https://docs.github.com/en/get-started/writing-on-github/getting-started-with-writing-and-formatting-on-github/basic-writing-and-formatting-syntax)
- [More detailed tutorial](https://www.datacamp.com/tutorial/markdown-in-jupyter-notebook)

By the way: In Markdown cells, you can also use simple HTML (e.g., `<key>...</key>` blocks) to gain more control over formatting.





### **Display images**

You may want to display your data, if possible. Here are a few ways to do that:

An easy method for displaying 2D images is using the [**Pillow**](https://pillow.readthedocs.io/en/stable/) library:

In [ ]:
# Option 1: Display an image with Pillow
path = "../data/images/kidney-cells-lowres.jpg"
image = PIL.Image.open(path)
display(image)                  

If you are working with the data as a NumPy array (e.g., when using the OpenCV interface), you can also display it using **matplotlib**.

In [ ]:
# Option 2: OpenCV / Matplotlib
path = "../data/images/ct-brain-slices.jpg"
image = cv.imread(path)
plt.imshow(image)
plt.axis("off")
plt.show()

For convenience, we also provide functions `display_image()` and `show_image()` the **tools** library, which includes various utilities used throughout this course.

In [ ]:
# Option 3a: Directly display a file
path = "../data/images/kidney-cells.jpg"
tools.display_image(path, scale=0.5)

# Option 3b: Similar, but with more options (e.g. title, frame, etc.)
tools.show_image(path, scale=0.5, 
                 title="Kidney cells", 
                 title_kwargs=dict(size=16, weight="bold", color="#7E487A"))

<br>
<br>


### **Display overlays**

When segmenting images, you may want to overlay the input image with the segmentation mask.  
There are several ways to do this ﷿﷿﷿ here are a few ideas:


In [ ]:
################################################
# Idea 1: Overlay a color on a grayscale image
################################################

# Enforce a (3-channel) color image
path_image = "../data/images/neurons-cultured.jpg"
image = cv.imread(path_image, cv.IMREAD_COLOR)
image = cv.cvtColor(image, cv.COLOR_BGR2RGB)

# Mask image
path_mask = "../data/images/neurons-cultured-mask.png"
mask = cv.imread(path_mask, cv.IMREAD_GRAYSCALE)

# Create overlay (RGB)
overlay_color = [255, 0, 0]
overlay_alpha = 0.3
overlay = image.copy()
overlay[mask > 0] = overlay_color
overlay = cv.addWeighted(image, 1 - overlay_alpha, overlay, overlay_alpha, 0)

# Display the images next to each other using a convenience function
tools.show_image_chain((image, overlay), titles=("Input", "Overlay"))

In [65]:
################################################
# Idea 2: Overlay contours on a grayscale image
################################################

overlay_color = [255, 255, 0]
line_width = 1
contours, _ = cv.findContours(mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
image_contours = image.copy()
cv.drawContours(image_contours, contours, -1, overlay_color, line_width)
tools.show_image_chain((image, image_contours), titles=("Input", "Contours"))

error: OpenCV(4.13.0) :-1: error: (-5:Bad argument) in function 'findContours'
> Overload resolution failed:
>  - image data type = bool is not supported
>  - Expected Ptr<cv::UMat> for argument 'image'


An advanced example: We can colorize different contours using distinct colors.

#### ***Strategy:***
- Use connected component labeling to assign a unique integer label to each region.
- Map each label to a different color by encoding it in the hue channel (in the HSV color space).
- Extract contours from the mask (ensure the mask is binary).
- Draw the contours with their assigned colors onto the original image.


In [ ]:
################################################
# Idea 3: Use colorized contours
################################################

# This will contain the result
image_contours = image.copy()

# Compute the "connected components" (= separate objects in the mask)
n_labels, labels = cv.connectedComponents(mask)

# Assign a different color to each label in the hue channel (HSV color space)
hue = np.uint8(150*labels/np.max(labels))
blank = 255*np.ones_like(hue)
labels = cv.merge([hue, blank, blank])

# Convert from HSV color space to RGB
labels = cv.cvtColor(labels, cv.COLOR_HSV2RGB)
# Set the background label (labels==0) to black
labels[labels==0] = 0

#﷿﷿Create a mask of the contours
line_width = 1
contours, _ = cv.findContours(mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
mask_contours = cv.drawContours(np.zeros_like(mask), contours, -1, 255, line_width)

# Assign the colored labels only along the contours
image_contours[mask_contours>0] = labels[mask_contours>0]

# Display the result
tools.show_image_chain((image, image_contours), titles=("Input", "Labeled contours"))

<br>
<br>

### **How to convert a Jupyter notebook into a HTML:**

- Don't forget to **save your notebook** before converting!
- Install the conversion tool (if not already installed): `pip install nbconvert`
- Convert the notebook to an HTML file: `jupyter nbconvert --to html main.ipynb`  
  The HTML file will be saved in the same folder as your notebook.

In [ ]:
# Make sure you save this notebook, otherwise the HTML 
# output will not contain the latest version!!

# Make sure you have nbcovnert installed
# If not, you can install it with pip:
#       pip install nbconvert --quiet
# Save the notebook as HTML
#       jupyter nbconvert --to html main.ipynb